# 초기 100사이클 기반 배터리 총수명 예측

이 노트북은 `main.py`와 같은 공통 단계 코드를 순서대로 호출합니다.

```text
config.py
    ↓
ExperimentRunner(CONFIG)
    ↓
단계별 셀: 데이터 준비 → 피처 계산 → 단계별 CV → 최종 학습 → 평가
```

- 실행 설정은 노트북에 복사하지 않고 `config.py`에서 읽습니다.
- 각 단계는 해당 절의 코드 셀을 실행할 때 수행됩니다.
- 같은 단계 셀을 다시 실행하면 완료된 상태를 재사용하여 중복 학습하지 않습니다.
- Batch3는 읽지 않습니다.
- 결과 CSV·모델·그래프 파일을 저장하지 않습니다.
- 실행 결과는 Batch2를 이미 확인한 후속 탐색 실험입니다.

## 1. 공통 코드와 설정 불러오기

노트북 위치를 기준으로 프로젝트 루트를 찾습니다. 개인 컴퓨터의 절대경로는 사용하지 않습니다.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "config.py").is_file():
    candidates = [
        parent / "DS-MINI"
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]
        if (parent / "DS-MINI" / "config.py").is_file()
    ]
    if not candidates:
        raise FileNotFoundError("DS-MINI/config.py 위치를 확인하세요.")
    PROJECT_ROOT = candidates[0]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CONFIG
from src.experiment import ExperimentRunner
from src.reporting import (
    candidate_preview,
    feature_catalog_preview,
    feature_range_preview,
    gap_summary,
    group_error_preview,
    performance_summary,
    plot_results,
    quality_summary,
    stage_summary,
)

CONFIG.validate()
print("프로젝트 루트:", PROJECT_ROOT)
print("데이터 기본 위치:", CONFIG.data_dir)

## 2. 원본 읽기와 셀 분할

실험 상태 객체를 만들고 Batch1·2 원본을 읽은 뒤 Train 36셀, Valid 10셀, Batch2 39셀을 확정합니다.

설정을 변경하려면 이 셀을 수정하지 않고 `config.py`를 수정한 뒤 커널을 다시 시작합니다.

In [ ]:
runner = ExperimentRunner(CONFIG, progress=True)
runner.load_data_and_split()
result = runner.result()
print("원본 읽기와 셀 분할 완료")

## 3. 환경·설정·분할 확인

실제 적용된 버전과 설정, 셀 단위 분할을 확인합니다.

In [ ]:
display(result.environment)
display(result.settings)
display(
    result.split_assignments.groupby(["batch", "role"], sort=False)
    .size()
    .rename("셀 수")
    .reset_index()
)
display(result.cv_folds.sort_values(["validation_fold", "cell_id"]).head(10))

## 4. 데이터 품질과 초기 피처

Batch1 첫 사이클 빈 측정, 내부저항 0, 확인된 손상 온도와 충전 조건 해석 상태를 확인합니다. 셀을 삭제하거나 수명 라벨을 바꾸지 않습니다.

In [ ]:
runner.build_features()
result = runner.result()
display(quality_summary(result))
display(
    result.data_quality.groupby(["batch", "policy_status"], sort=False)
    .size()
    .rename("셀 수")
    .reset_index()
)
display(
    result.feature_quality.groupby("delta_q_status", sort=False)
    .size()
    .rename("셀 수")
    .reset_index()
)
display(feature_catalog_preview(result, n=10))

## 5. 넓은 1차 교차검증

전체 초기 피처군 × 모델 × 타깃을 기본 설정으로 평가합니다. 계산은 전체 후보로 수행하고, 표에는 성능과 구성 다양성을 반영한 대표 후보를 최대 10개만 표시합니다.

In [ ]:
runner.run_stage1()
result = runner.result()
display(stage_summary(result))
display(candidate_preview(result, "stage1", n=10))

## 6. 확장·변화 피처 비교

1차 생존 구성에서 상대 용량 변화, 내부저항 변화, 온도 변화, 기울기 교체, 평균 Qd 제외와 정규화 ΔQ 후보를 만듭니다. 부모 후보와 확장 후보를 같은 분기에서 비교합니다.

In [ ]:
runner.run_stage2()
result = runner.result()
display(candidate_preview(result, "stage2", n=10))

## 7. 생존 후보의 매개변수 재탐색

엘라스틱넷은 `alpha`·`l1_ratio`, 릿지는 `alpha`, 랜덤 포레스트는 깊이·말단 최소 표본 수를 비교합니다. 모든 후보는 같은 5개 fold를 사용합니다.

In [ ]:
runner.run_stage3()
result = runner.result()
display(candidate_preview(result, "stage3", n=10))
display(result.final_configurations)

## 8. 최종 학습·평가

학습 내부 CV로 선택한 구성, 기존 피처 대표, 변경 피처 대표와 평균 수명 기준선을 비교합니다. 각 대표 구성은 Train 36셀로 학습하고, 같은 모델로 Valid 10셀과 Batch2 39셀을 예측합니다.

Valid·Batch2 결과로 대표 구성을 바꾸지 않습니다.

In [ ]:
runner.run_final_evaluation()
result = runner.result()
display(performance_summary(result))
display(gap_summary(result))

## 9. Batch2 오류 분석

최종 구성의 단수명 과대 예측, 수명 구간·프로토콜별 오차와 큰 오차 셀을 확인합니다.

In [ ]:
runner.run_error_analysis()
result = runner.result()
display(result.error_analysis["short_life_summary"])
display(group_error_preview(result, n=10))
display(
    result.error_analysis["worst_absolute"][
        [
            "cell_id", "actual", "predicted", "signed_error",
            "absolute_error", "ape_pct", "charging_policy",
        ]
    ]
)
display(feature_range_preview(result, n=10))

## 10. 결과 그래프

공통 결과 객체에서 그래프를 생성합니다. 파일로 저장하지 않습니다.

In [ ]:
figures = plot_results(result)
for figure in figures:
    display(figure)
    plt.close(figure)

## 11. 실행 결과의 범위

- 이 노트북과 `main.py`는 같은 설정·분할·후보·평가 코드를 사용합니다.
- 노트북은 결과를 표시하기 위해 모델을 다시 학습하지 않습니다.
- Batch3는 데이터 폴더에 있어도 처리하지 않습니다.
- Batch2를 이미 확인한 후속 작업이므로 결과는 탐색 실험입니다.
- 9.1%는 비교 목표이며 동일 분할의 논문 재현 성능이 아닙니다.
- 단계적 선별은 탐색량을 줄이는 방법이며 전체 조합의 최적해를 보장하지 않습니다.
- 원본 데이터, 결과표, 모델과 그래프를 별도 파일로 저장하지 않습니다.